**Configuración de GPU**

In [ ]:
import torch
import os

# Verificación de GPU
print("="*70)
print(" Configuración de GPU")
print("="*70)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(" WARNING: No se detectó GPU. El entrenamiento será lento.")

print("="*70)

**Explorar Dataset**

In [ ]:
import matplotlib.pyplot as plt
import random
from PIL import Image
import glob

# Carpeta con las imágenes
IMG_FOLDER = '/ruta_a_las_imagenes'

# Muestra 5 perros aleatorios
subfolders = [f.path for f in os.scandir(IMG_FOLDER) if f.is_dir()]
random_dogs = random.sample(subfolders, min(5, len(subfolders)))

fig, axes = plt.subplots(5, 5, figsize=(15, 15))
fig.suptitle('Muestras del Dataset (5 perros, 5 imágenes c/u)',
             fontsize=16, fontweight='bold')

for i, dog_folder in enumerate(random_dogs):
    images = glob.glob(os.path.join(dog_folder, '*.png'))

    if len(images) > 0:
        sample_images = random.sample(images, min(5, len(images)))

        for j, img_path in enumerate(sample_images):
            img = Image.open(img_path)
            axes[i, j].imshow(img)
            axes[i, j].axis('off')

            if j == 0:
                dog_id = os.path.basename(dog_folder)
                axes[i, j].set_title(f'ID: {dog_id}', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

**Instalar librerías**

In [ ]:
!pip install -q lightly
!pip install -q scikit-learn
!pip install -q timm

print(" Librerías instaladas correctamente")

**Configuración de Parámetros**

In [ ]:
class Config:
    """Configuración del entrenamiento DINO"""

    # Rutas
    DATA_PATH = '/ruta_a_dataset'
    TRAIN_CSV = 'Splits'
    OUTPUT_DIR = '/output/history'
    MODEL_DIR = '/output/model'
    RESULTS_DIR = '/output/eval_results'

    # Modelo
    BACKBONE = 'resnet50'  # Opciones: resnet50, tiny_vit
    HIDDEN_DIM = 1024
    OUTPUT_DIM = 4096

    # Entrenamiento
    BATCH_SIZE = 128  # Ajustar según memoria RAM de GPU
    NUM_WORKERS = 2
    EPOCHS = 100
    SAVE_EVERY = 10

    # Learning Rate 
    BASE_LR = 0.0005
    MIN_LR = 1e-6
    WARMUP_EPOCHS = 10
    WEIGHT_DECAY = 0.04
    WEIGHT_DECAY_END = 0.4

    # Data Augmentation
    GLOBAL_CROP_SIZE = 224
    LOCAL_CROP_SIZE = 96
    N_LOCAL_VIEWS = 4
    GLOBAL_CROP_SCALE = (0.4, 1.0)
    LOCAL_CROP_SCALE = (0.05, 0.4)

    # Configuración Dino
    WARMUP_TEACHER_TEMP = 0.04
    TEACHER_TEMP = 0.07
    WARMUP_TEACHER_TEMP_EPOCHS = 20 # Es recomendable usar 20 o 30
    MOMENTUM_TEACHER_START = 0.996
    MOMENTUM_TEACHER_END = 1.0

    # === CHECKPOINT PARA CONTINUAR ===
    RESUME_CHECKPOINT = None # Cambiar a '/output/history' si se quiere continuar desde un punto de guardado

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

config = Config()

# Mostrar configuración
print(" Configuración del Entorno")
print("="*70)
print(f"Backbone: {config.BACKBONE}")
print(f"Batch Size: {config.BATCH_SIZE}")
print(f"Épocas: {config.EPOCHS}")
print(f"Learning Rate: {config.BASE_LR}")
print(f"Output Dir: {config.OUTPUT_DIR}")
print(f"Device: {config.DEVICE}")
print("="*70)

**Cargar Dataset con CSV**

In [ ]:
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image

class Dataset_CSV(Dataset):
    """Carga de dataset con soporte para CSV"""

    def __init__(self, csv_path, root_dir, transform=None):
        """
        Args:
            csv_path: Ruta al train.csv
            root_dir: Directorio raíz donde están las imágenes
            transform: Transformaciones (manejadas por DINOCollateFunction)
        """
        self.df = pd.read_csv(csv_path)
        self.root_dir = root_dir
        self.transform = transform

        # Verificar que existan las columnas necesarias
        if 'filename' not in self.df.columns:
            raise ValueError("El CSV debe contener la columna 'filename'")

        print(f" Dataset cargado:")
        print(f"   - Total imágenes: {len(self.df)}")
        if 'label' in self.df.columns:
            print(f"   - Individuos únicos: {self.df['label'].nunique()}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        """Retorna imagen sin transformar (las hace DINOCollateFunction)"""
        relative_path = self.df.iloc[idx]['filename']
        full_path = os.path.join(self.root_dir, relative_path)

        if not os.path.exists(full_path):
            raise FileNotFoundError(f"Imagen no encontrada: {full_path}")

        img = Image.open(full_path).convert("RGB")

        # Retornar formato esperado por lightly
        # (imagen, label=0, metadata={})
        return img, 0, {}

**Modelo y Arquitectura**

In [ ]:
import copy
import torch
import torchvision
import pandas as pd
import numpy as np
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
import timm

# Lightly para DINO
import lightly.data
from lightly.data.collate import DINOCollateFunction
from lightly.loss import DINOLoss
from lightly.models.modules import DINOProjectionHead
from lightly.utils.scheduler import cosine_schedule
from lightly.models.utils import update_momentum

def build_backbone(arch='resnet50', pretrained=False):
    """Construye el backbone según la arquitectura especificada"""

    if arch == 'resnet50':
        backbone = torchvision.models.resnet50(
            weights='IMAGENET1K_V1' if pretrained else None
        )
        backbone_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()

    elif arch == 'tiny_vit':
        backbone = timm.create_model(
            'tiny_vit_5m_224',
            pretrained=False,
            num_classes=0
        )
        backbone_dim = backbone.num_features

    else:
        raise ValueError(f"Arquitectura no soportada: {arch}")

    return backbone, backbone_dim

def build_model(config):
    """Construye el modelo completo (student + teacher)"""

    print("\n Construyendo modelos...")

    # Backbone
    backbone, backbone_dim = build_backbone(
        config.BACKBONE,
        pretrained=False  # Entrenar desde cero
    )

    print(f" Backbone: {config.BACKBONE} (dim={backbone_dim})")

    # Projection Head
    head_config = {
        "input_dim": backbone_dim,
        "hidden_dim": config.HIDDEN_DIM,
        "output_dim": config.OUTPUT_DIM,
        "batch_norm": True,
        "freeze_last_layer": 1,  # Freeze en primera época
    }

    print(f" Projection Head: {backbone_dim} → {config.HIDDEN_DIM} → {config.OUTPUT_DIM}")

    # Student
    student_backbone = backbone.to(config.DEVICE)
    student_head = DINOProjectionHead(**head_config).to(config.DEVICE)

    # Teacher (copia del student)
    teacher_backbone = copy.deepcopy(student_backbone)
    teacher_head = DINOProjectionHead(**head_config)
    teacher_head.load_state_dict(student_head.state_dict())
    teacher_head = teacher_head.to(config.DEVICE)

    # Congelar teacher
    for param in teacher_backbone.parameters():
        param.requires_grad = False
    for param in teacher_head.parameters():
        param.requires_grad = False

    print(f" Teacher congelado (sin gradientes)")

    return student_backbone, student_head, teacher_backbone, teacher_head

**Funciones de Checkpoint**

In [ ]:
def save_checkpoint(path, epoch, student_backbone, student_head,
                   teacher_backbone, teacher_head, optimizer, scaler,
                   loss_history, config):
    """Guarda checkpoint completo"""

    checkpoint = {
        "epoch": epoch,
        "student_backbone": student_backbone.state_dict(),
        "student_head": student_head.state_dict(),
        "teacher_backbone": teacher_backbone.state_dict(),
        "teacher_head": teacher_head.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "loss_history": loss_history,
        "config": vars(config),
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint guardado: {path}")

def load_checkpoint(path, student_backbone, student_head,
                   teacher_backbone, teacher_head, optimizer, scaler, config):
    """Carga checkpoint para continuar entrenamiento"""

    if not os.path.exists(path):
        print(f"  Checkpoint no encontrado: {path}")
        return 0, []

    print(f" Cargando checkpoint: {path}")
    checkpoint = torch.load(path, map_location=config.DEVICE, weights_only=False)

    student_backbone.load_state_dict(checkpoint["student_backbone"])
    student_head.load_state_dict(checkpoint["student_head"])
    teacher_backbone.load_state_dict(checkpoint["teacher_backbone"])
    teacher_head.load_state_dict(checkpoint["teacher_head"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scaler.load_state_dict(checkpoint["scaler"])

    epoch = checkpoint["epoch"] + 1
    loss_history = checkpoint.get("loss_history", [])

    print(f" Checkpoint cargado. Reanudando desde época {epoch}")

    return epoch, loss_history

**Entrenamiento**

In [ ]:
def train_dino(config):
    """Loop principal de entrenamiento DINO"""

    # Preparar dataset
    print("\n Preparando dataset...")

    dataset = lightly.data.LightlyDataset(input_dir= config.DATA_PATH)

    # Collate function (multi-crop augmentation)
    collate_fn = DINOCollateFunction(
        global_crop_size=config.GLOBAL_CROP_SIZE,
        local_crop_size=config.LOCAL_CROP_SIZE,
        n_local_views=config.N_LOCAL_VIEWS,
        global_crop_scale=config.GLOBAL_CROP_SCALE,
        local_crop_scale=config.LOCAL_CROP_SCALE,
    )

    # DataLoader
    dataloader = DataLoader(
        dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=config.NUM_WORKERS,
        collate_fn=collate_fn,
        pin_memory=True,
        persistent_workers=True if config.NUM_WORKERS > 0 else False
    )

    print(f" DataLoader listo: {len(dataloader)} batches por época")

    # Modelos
    student_backbone, student_head, teacher_backbone, teacher_head = build_model(config)

    # Función de Pérdida
    criterion = DINOLoss(
        output_dim=config.OUTPUT_DIM,
        warmup_teacher_temp=config.WARMUP_TEACHER_TEMP,
        teacher_temp=config.TEACHER_TEMP,
        warmup_teacher_temp_epochs=config.WARMUP_TEACHER_TEMP_EPOCHS,
        student_temp=0.1,
    ).to(config.DEVICE)

    print(f" DINO Loss configurado")

    # Optimizador
    params = list(student_backbone.parameters()) + list(student_head.parameters())
    optimizer = AdamW(params, lr=config.BASE_LR, weight_decay=config.WEIGHT_DECAY)

    # AMP Scaler
    scaler = torch.amp.GradScaler()

    # Learning rate scheduler
    num_batches = len(dataloader)
    total_steps = config.EPOCHS * num_batches
    warmup_steps = config.WARMUP_EPOCHS * num_batches

    # Crear schedules manualmente para cada step
    # La función cosine_schedule de lightly se llama por step
    def get_lr_at_step(step):
        """Learning rate con warmup lineal + cosine decay"""
        if step < warmup_steps:
            # Warmup lineal
            return config.BASE_LR * (step / warmup_steps)
        else:
            # Cosine decay desde BASE_LR hasta MIN_LR
            return cosine_schedule(
                step=step - warmup_steps,
                max_steps=total_steps - warmup_steps,
                start_value=config.BASE_LR,
                end_value=config.MIN_LR
            )

    def get_wd_at_step(step):
        """Weight decay con cosine schedule (sin warmup)"""
        return cosine_schedule(
            step=step,
            max_steps=total_steps,
            start_value=config.WEIGHT_DECAY,
            end_value=config.WEIGHT_DECAY_END
        )

    # Carga el checkpoint si existe
    start_epoch = 0
    loss_history = []

    if config.RESUME_CHECKPOINT:
        start_epoch, loss_history = load_checkpoint(
            config.RESUME_CHECKPOINT,
            student_backbone, student_head,
            teacher_backbone, teacher_head,
            optimizer, scaler, config
        )

    print("\n" + "="*70)
    print(" INICIANDO ENTRENAMIENTO")
    print("="*70)

    for epoch in range(start_epoch, config.EPOCHS):
        student_backbone.train()
        student_head.train()

        epoch_loss = 0.0
        progress_bar = tqdm(dataloader, desc=f"Época {epoch+1}/{config.EPOCHS}")

        for batch_idx, (views, _, _) in enumerate(progress_bar):

            # Calcular step global
            global_step = epoch * num_batches + batch_idx
            current_lr = get_lr_at_step(global_step)
            current_wd = get_wd_at_step(global_step)

            # Actualizar LR y WD según schedule
            for param_group in optimizer.param_groups:
                param_group["lr"] = current_lr
                param_group["weight_decay"] = current_wd

            # Momentum schedule para teacher
            momentum = cosine_schedule(
                step=global_step,
                max_steps=total_steps,
                start_value=config.MOMENTUM_TEACHER_START,
                end_value=config.MOMENTUM_TEACHER_END
            )

            views = [v.to(config.DEVICE) for v in views]
            global_views = views[:2]
            local_views = views[2:] if len(views) > 2 else []

            # AMP
            with torch.amp.autocast(device_type='cuda'):
                # Teacher forward (sin gradientes)
                with torch.no_grad():
                    teacher_out = []
                    for v in global_views:
                        features = teacher_backbone(v)
                        out = teacher_head(features)
                        teacher_out.append(out)

                # Student forward
                student_out = []
                for v in global_views + local_views:
                    features = student_backbone(v)
                    out = student_head(features)
                    student_out.append(out)

                # Calcular loss
                loss = criterion(teacher_out, student_out, epoch=epoch)

            # Backward pass
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # Actualizar teacher con EMA
            update_momentum(student_backbone, teacher_backbone, m=momentum)
            update_momentum(student_head, teacher_head, m=momentum)

            # Logging
            epoch_loss += loss.item()
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'lr': f'{current_lr:.6f}',
                'mom': f'{momentum:.4f}'
            })

        # Promedio de loss de la época
        avg_loss = epoch_loss / num_batches
        loss_history.append(avg_loss)

        print(f"\n Época {epoch+1}/{config.EPOCHS} - Loss promedio: {avg_loss:.4f}")

        # Guardar checkpoint
        if (epoch + 1) % config.SAVE_EVERY == 0 or (epoch + 1) == config.EPOCHS:
            checkpoint_path = os.path.join(
                config.OUTPUT_DIR,
                f'checkpoint_epoch_{epoch+1:03d}.pth'
            )
            save_checkpoint(
                checkpoint_path,
                epoch,
                student_backbone, student_head,
                teacher_backbone, teacher_head,
                optimizer, scaler,
                loss_history, config
            )

            # También guardar como "último"
            last_checkpoint_path = os.path.join(config.OUTPUT_DIR, 'checkpoint_last.pth')
            save_checkpoint(
                last_checkpoint_path,
                epoch,
                student_backbone, student_head,
                teacher_backbone, teacher_head,
                optimizer, scaler,
                loss_history, config
            )

    # Guardar modelo final
    final_backbone_path = os.path.join(config.MODEL_DIR, 'dino_model_final.pth')
    torch.save(student_backbone.state_dict(), final_backbone_path)
    print(f"\n Modelo final guardado: {final_backbone_path}")

    # Plot de pérdida
    plot_loss_curve(loss_history, config.RESULTS_DIR)

    return student_backbone, student_head

def plot_loss_curve(loss_history, output_dir):
    """Grafica la curva de pérdida"""

    plt.figure(figsize=(10, 6))
    plt.plot(loss_history, linewidth=2)
    plt.xlabel('Época', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Curva de Entrenamiento DINO', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plot_path = os.path.join(output_dir, 'training_loss.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f" Gráfica guardada: {plot_path}")
    plt.show()

**Iniciar Entrenamiento**

In [ ]:
# IMPORTANTE: El entrenamiento puede tomar varias horas

print("\n" + "="*70)
print(" INICIANDO ENTRENAMIENTO DINO")
print("="*70)
print("  IMPORTANTE:")
print("  - Esto tomará varias horas (depende de EPOCHS)")
print("  - Los checkpoints se guardan automáticamente")
print("  - Si se interrumpe, puedes continuar desde el último checkpoint")
print("="*70)

# Entrenar
student_backbone, student_head = train_dino(config)

print("\n ENTRENAMIENTO COMPLETADO")